# Theme extraction

## 0. A small introduction to theme extraction

To extract themes from both the contest images (via their textual descriptions) and the user-submitted captions in a scalable and linguistically meaningful way, we rely on **sentence-level embeddings** rather than traditional bag-of-words topic models. Our pipeline is presented like so:

- Sentence-BERT embeddings
- Unsupervised clustering
- Theme assignment (for both images and captions)

Each description or caption is converted into a dense vector representation using a **Sentence-BERT** model, which captures contextual meaning, humor, and implicit relationships far better than word-frequency-based methods. Our approach is particularly well suited for caption data, which is typically short, creative, and semantically rich.

We initially considered classical topic modeling approaches like **Latent Dirichlet Allocation (LDA)**, but LDA’s bag-of-words assumptions and poor performance on short, diverse, humor-oriented texts make it unsuitable for this dataset. We also evaluated **BERTopic**, which combines embeddings with topic modeling, but its reliance on UMAP makes it difficult to scale efficiently to the size of our dataset.

Once all texts have been embedded into a shared semantic space, we apply **unsupervised clustering** methods (e.g., MiniBatchKMeans, HDBSCAN) to group semantically similar texts into coherent themes that reflect recurring patterns across contests. By clustering all descriptions and captions globally, rather than per contest, we create a unified thematic space that remains consistent throughout the dataset—an essential property for cross-contest comparisons and downstream analyses.

This embedding-based clustering pipeline enables us to assign each caption and each contest image to a specific **theme_id**, producing a structured and interpretable representation of the humor of the captions and visual concepts present in the dataset. These theme assignments provide the foundation for following analyses such as exploring funniness-theme correlations, measuring image–caption alignment, and visualizing global humor dynamics.

In [1]:
import json
from pathlib import Path

from tqdm.auto import tqdm

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import MiniBatchKMeans

In [2]:
# General variables: paths
ROOT = Path("../../data/newyorker_caption_contest")
CONTESTS_JSON = ROOT / "contests.json"
CSV_DIR = ROOT / "cleaned_data"

# Output directory management: create a new output directory if needed
output_dir_base = "themes_extraction_csv"
base_path = ROOT / output_dir_base

# Start by considering the base path and then suffixed paths
candidate_idx = None  # None means base, otherwise use _{candidate_idx}
while True:
    candidate = base_path if candidate_idx is None else ROOT / f"{output_dir_base}_{candidate_idx}"
    if not candidate.exists():
        # Not present: create and use it
        OUTPUT_DIR = candidate
        break
    if not any(candidate.iterdir()):  # exists and empty: reuse it
        OUTPUT_DIR = candidate
        break
    # exists and non-empty: try next suffix
    candidate_idx = 1 if candidate_idx is None else candidate_idx + 1
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Sentence-BERT model
BERT_MODEL = "sentence-transformers/all-mpnet-base-v2"

# print paths
print(f"Root: {ROOT.resolve()}")
print(f"Contests JSON: {CONTESTS_JSON.resolve()}")
print(f"CSV data folder: {CSV_DIR.resolve()}")
print(f"Output folder: {OUTPUT_DIR.resolve()}")  
print(f"\nBERT model: {BERT_MODEL}")

Root: /Users/jeremyserillon/Desktop/ada-2025-project-json-derulo/data/newyorker_caption_contest
Contests JSON: /Users/jeremyserillon/Desktop/ada-2025-project-json-derulo/data/newyorker_caption_contest/contests.json
CSV data folder: /Users/jeremyserillon/Desktop/ada-2025-project-json-derulo/data/newyorker_caption_contest/cleaned_data
Output folder: /Users/jeremyserillon/Desktop/ada-2025-project-json-derulo/data/newyorker_caption_contest/themes_extraction_csv_2

BERT model: sentence-transformers/all-mpnet-base-v2


## 1. Loading the data

We start by loading our data in a unified way in a custom pandas dataframe.

In [ ]:

def load_contests(contests_json_path: Path):
    with open(contests_json_path, "r", encoding="utf-8") as f:
        contests = json.load(f)
    return contests

def build_docs_dataframe(debug=False):
    contests = load_contests(CONTESTS_JSON)
    docs = []
    
    error_csv = []

    # tqdm over contests
    for contest in tqdm(contests, desc="Step 1: data loading/building"):
        cid = contest["contest_id"]
        
        # print DEBUG
        # print(f"Processing contest ID: {cid}")

        raw_meta = contest.get("metadata")
        meta = raw_meta or {}   # normalize None -> {}

        # ---- Build one aggregated "image document" per contest ----
        parts = []

        for key in ["image_descriptions", "image_uncanny_descriptions",
                    "image_locations", "questions"]:
            values = meta.get(key) or []
            parts.extend(values)

        entities = meta.get("entities") or []
        for url in entities:
            last = str(url).rstrip("/").split("/")[-1]
            parts.append(last.replace("_", " "))

        image_text = " ".join(map(str, parts)).strip()

        if image_text:
            docs.append(
                {
                    "doc_id": f"image_{cid}",
                    "contest_id": cid,
                    "kind": "image",
                    "text": image_text,
                }
            )

        # ---- Add caption documents for this contest ----
        csv_path = CSV_DIR / f"{cid}.csv"
        if not csv_path.exists():
            csv_path = CSV_DIR / f"{cid}_cleaned.csv"
            if not csv_path.exists():
                print(f"Warning: CSV for contest {cid} not found at {CSV_DIR / f'{cid}.csv'} and neither at {csv_path}.")
                continue

        try:
            captions_df = pd.read_csv(csv_path)
        except Exception as e:
            print(f"Error reading CSV for contest {cid} at {csv_path}: {e}")
            error_csv.append(cid)
            continue

        # optional inner tqdm if you want per-caption progress too
        for idx, row in captions_df.iterrows():
            cap_text = str(row.get("caption", "")).strip()
            if not cap_text:
                continue

            docs.append(
                {
                    "doc_id": f"caption_{cid}_{idx}",
                    "contest_id": cid,
                    "kind": "caption",
                    "text": cap_text,
                    "rank": row.get("rank", None),
                    "mean": row.get("mean", None),
                    "precision": row.get("precision", None),
                    "votes": row.get("votes", None),
                    "not_funny": row.get("not_funny", None),
                    "somewhat_funny": row.get("somewhat_funny", None),
                    "funny": row.get("funny", None),
                }
            )

    df = pd.DataFrame(docs)
    
    # keep only expected first 1000 rows for debugging
    if debug:
        df = df.head(5000)
    
    return df, error_csv


# other loading function to load from an output csv that comes from this notebook
def load_docs_with_clean_text(docs_csv_path, debug=False):
    """
    Load the prebuilt docs_with_clean_text.csv file so you don’t have to rebuild
    docs from individual contest files. Returns the dataframe plus an empty
    error list for API parity with build_docs_dataframe.
    """
    docs_csv_path = Path(docs_csv_path)
    if not docs_csv_path.exists():
        raise FileNotFoundError(f"docs_with_clean_text.csv not found at {docs_csv_path}")

    cols = [
        "doc_id", "contest_id", "kind", "text",
        "rank", "mean", "precision", "votes",
        "not_funny", "somewhat_funny", "funny",
        "clean_text",
    ]
    df = pd.read_csv(docs_csv_path)
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"docs_with_clean_text.csv is missing expected columns: {missing}")
    
    # remove rows with NAN (the embedding step cannot handle those)
    # df["text"] = df["text"].fillna("")
    
    # keep only expected first 1000 rows for debugging
    if debug:
        df = df.head(5000)

    return df, []

In [ ]:
# To load from the original contest files put False
#   => uses CSV_DIR which contains all csv files
# To load from a the prebuilt CSV (that comes from this notebook) put True
#   => uses a docs_with_themes.csv formated file (below step 5.3 and 6)
load_from_docs_csv = True 
use_clean_text = True       # use replace the text column by the clean one

debug = False   # only consider first 1000 rows if True
                # if False, process all rows (~ 2.3 million rows)


if load_from_docs_csv:
    original_docs_df, error_csv = load_docs_with_clean_text("../../data/newyorker_caption_contest/themes_extraction_csv_1/docs_with_clean_text.csv", debug=debug)
    
    docs_df = original_docs_df.copy()
    
    if use_clean_text:
        # replace "text" column with "clean_text" column
        docs_df["text"] = docs_df["clean_text"]
        # remove "clean_text" column
        docs_df = docs_df.drop(columns=["clean_text"])
        # remove rows with NAN (the embedding step cannot handle those)
        docs_df["text"] = docs_df["text"].fillna("")
        
else:
    original_docs_df, error_csv = build_docs_dataframe(debug=debug)
    docs_df = original_docs_df.copy()

print("Total documents:", len(docs_df))
print(docs_df["kind"].value_counts())

if error_csv:
    print("Contests with CSV read errors:", error_csv)

Total documents: 2292901
kind
caption    2292661
image          240
Name: count, dtype: int64


In [ ]:
original_docs_df.head()

In [ ]:
if use_clean_text:
    docs_df.head()

,doc_id,contest_id,kind,text,rank,mean,precision,votes,not_funny,somewhat_funny,funny
0,image_510,510,image,man relax city street go business step pass ma...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,caption_510_0,510,caption,congressman obstruction job,0.0,1.913043,0.094022,69.0,24.0,27.0,18.0
2,caption_510_1,510,caption,mean middle class step,1.0,1.842105,0.191381,19.0,8.0,6.0,5.0
3,caption_510_2,510,caption,suit look flat,2.0,1.711111,0.112915,45.0,21.0,16.0,8.0
4,caption_510_3,510,caption,right woman come know,3.0,1.625000,0.116657,32.0,15.0,14.0,3.0


## 2. Sentence-BERT encoding

This step will take a lot of time to process.

So go grab a coffee, open a good book, or a Selling Sunset episode ... it'll definitely speed up the process. Or invent time travelling, maybe that would work better... maybe... who knows...

<!-- oooor I could tell you about some gossip I heard the other day at the cafetaria! Aaaah, now you're interested. 

""" <br>
So basically, a friend of mine broke up with her boyfriend and slept with his guy who she had the biggest crush on for YEARS. A few days later she started getting a burning in her vagina. She thought it was a yeast infection. NOPE! It was herpes.

However, the guy she slept with had never had a herpes outbreak prior to the current one so his herpes was new too. Where did guy A get the herpes from? He slept with a woman who had a recent herpes outbreak. This woman also slept with guy A's FATHER and is now pregnant!

But where did the herpes originally come from? From guy A's father.

So to sum up this woman is now pregnant, they don't know if the childs father is father or son, and now everyone has herpes. 
<br>""" -->

In [6]:
model = SentenceTransformer(BERT_MODEL)

def compute_embeddings(texts, batch_size=64):
    # returns a numpy array of shape (N, D)
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # helpful for cosine similarity
    )
    return embeddings

embeddings = compute_embeddings(docs_df["text"].tolist())
print("Embeddings shape:", embeddings.shape)


Batches:   0%|          | 0/35827 [00:00<?, ?it/s]

Embeddings shape: (2292901, 768)


## 3. Global clustering (across all captions + images)

We’ll do global clustering across all documents (captions + images), so they share the same theme space.

For scalability, we will use MiniBatchKMeans. To start we will choose a number of clusters of ```n_clusters=80``` and fine tune it afterwards.

In [ ]:
N_CLUSTERS = 80

# apply clustering
kmeans = MiniBatchKMeans(
    n_clusters=N_CLUSTERS,
    batch_size=2048,
    max_iter=100,
    random_state=42,
)
kmeans.fit(embeddings)

theme_labels = kmeans.predict(embeddings)

# save results to dtaframe
docs_df["theme_id"] = theme_labels
docs_df.head()

# copy for good practive
docs_with_themes = docs_df.copy()


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## 4. Theme definitions: TF–IDF-based labels per theme_id

Now that we have groups of themes, we want to find what describeds each of them accordingly by finding keywords to described each one of them.

Idea:
- Fit a single TF–IDF over all texts.
- For each theme (cluster), compute the average TF–IDF vector over its documents.
- Take the top N terms → that’s your label candidate.

### 4.1. Fit TF–IDF on all docs

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

# fine-tune 
MAX_FEATURES = 100000   # influence the size of the vocabulary => larger = more precise but slower
MIN_DF = 5              # ignore words that appear in <5 documents
NGRAM_RANGE = (1, 3)    # unigrams + bigrams, example: "new york"

vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES,
    min_df=MIN_DF,
    ngram_range=NGRAM_RANGE,
    stop_words="english"
)

tfidf_matrix = vectorizer.fit_transform(docs_with_themes["text"])
feature_names = np.array(vectorizer.get_feature_names_out())

print(tfidf_matrix.shape)  # (n_docs, n_features)


(2292901, 100000)


### 4.2. Get top TF–IDF terms per cluster

We’ll compute, for each theme_id:
- the mean TF–IDF vector across its docs
- the top-k features with highest mean score (we'll take top-k = 10)

In [10]:
from collections import defaultdict

def get_top_terms_per_theme(docs_df, tfidf_matrix, feature_names, top_k):
    """
    For each theme in docs_df, compute the top_k terms with highest mean TF-IDF    
    """
    
    theme_to_terms = {}
    theme_ids = sorted(docs_df["theme_id"].unique())

    for theme_id in theme_ids:
        mask = (docs_df["theme_id"] == theme_id).values
        if mask.sum() == 0:
            continue

        # slice TF-IDF rows for this theme
        theme_tfidf = tfidf_matrix[mask]

        # compute mean tf-idf across docs in this theme (1 x n_features)
        mean_tfidf = theme_tfidf.mean(axis=0)  # still a sparse matrix
        mean_tfidf = np.asarray(mean_tfidf).ravel()

        # get top_k indices
        if np.all(mean_tfidf == 0):
            # cluster with weird texts, skip
            theme_to_terms[theme_id] = []
            continue

        top_idx = mean_tfidf.argsort()[::-1][:top_k]
        top_terms = feature_names[top_idx].tolist()
        theme_to_terms[theme_id] = top_terms

    return theme_to_terms

In [11]:
# Run the top terms extraction
theme_to_terms = get_top_terms_per_theme(
    docs_with_themes, tfidf_matrix, feature_names, top_k=10
)

# display first 7 themes with 5 examples of a doc text containing the theme id
for id in list(theme_to_terms.keys()):
    print(f"Theme id {id}:   {theme_to_terms[id]}\nPrompt examples:")
    # print 5 examples of captions per themes:
    i = 0
    for idx, row in docs_with_themes[docs_with_themes["theme_id"] == id].iterrows():
        print(f"  - '{row['text']}'")
        i += 1
        if i >= 5:
            print("")
            break

Theme id 0:   ['gluten', 'vegan', 'gluten free', 'free', 'vegetarian', 'organic', 'gmo', 'lactose', 'intolerant', 'kosher']
Prompt examples:
  - 'yellow snow gluten free'
  - 'natural fiber'
  - 'food natural mat synthetic'
  - 'allow wool mat vegan studio'
  - 'certain mat grass feed'

Theme id 1:   ['time', 'minute', 'think time', 'clock', 'think', 'tell', 'hour', 'time tell', 'tell time', 'let']
Prompt examples:
  - 'time masochists anonymous meeting'
  - 'time stop smell concrete'
  - 'minute meter'
  - 'need time'
  - 'year step people time'

Theme id 2:   ['day', 'week', 'today', 'friday', 'tomorrow', 'weekend', 'date', 'casual', 'yesterday', 'casual friday']
Prompt examples:
  - 'day star wars'
  - 'day ass line'
  - 'sir flash mob thursday'
  - 'today report man street chuck'
  - 'mean doormat day real charity'

Theme id 3:   ['dog', 'cat', 'pet', 'bark', 'leash', 'animal', 'puppy', 'vet', 'know', 'think']
Prompt examples:
  - 'bravo pass apathy test fly color dog look concern'

In [ ]:
# Display the first 10 rows with their theme ids and top terms and text
for i in range(10):
    doc = docs_with_themes.iloc[i]
    theme_id = doc["theme_id"]
    top_terms = theme_to_terms.get(theme_id, [])
    print(f"{doc['doc_id']}: '{doc['text']}'\n => Theme ID {theme_id}: {top_terms}\n")


In [14]:
# list all themes with their top terms
for theme_id, top_terms in theme_to_terms.items():
    print(f"Theme ID {theme_id}: {top_terms}")
    
# count number of themes without any top terms
no_terms_count = sum(1 for terms in theme_to_terms.values() if not terms)
print(f"\nNumber of themes without any top terms: {no_terms_count}")

Theme ID 0: ['gluten', 'vegan', 'gluten free', 'free', 'vegetarian', 'organic', 'gmo', 'lactose', 'intolerant', 'kosher']
Theme ID 1: ['time', 'minute', 'think time', 'clock', 'think', 'tell', 'hour', 'time tell', 'tell time', 'let']
Theme ID 2: ['day', 'week', 'today', 'friday', 'tomorrow', 'weekend', 'date', 'casual', 'yesterday', 'casual friday']
Theme ID 3: ['dog', 'cat', 'pet', 'bark', 'leash', 'animal', 'puppy', 'vet', 'know', 'think']
Theme ID 4: ['bear', 'elephant', 'sheep', 'animal', 'tail', 'giraffe', 'howl', 'monkey', 'lion', 'wolf']
Theme ID 5: ['room', 'elephant', 'elephant room', 'table', 'couch', 'seat', 'chair', 'desk', 'ikea', 'sit']
Theme ID 6: ['say', 'honey', 'mean', 'hear', 'know', 'stop', 'tell', 'let', 'little', 'swear']
Theme ID 7: ['insurance', 'cover', 'policy', 'pay', 'warranty', 'plan', 'tax', 'insurance cover', 'health', 'life insurance']
Theme ID 8: ['thing', 'mean', 'guess', 'right', 'bad', 'guy', 'funny', 'point', 'wrong', 'little']
Theme ID 9: ['think',

Now let's pick one concise label per theme by blending TF‑IDF candidates with embedding similarity and a distinctiveness penalty.

1) cleans candidates
2) for each cluster computes its embedding centroid, embeds each candidate term
3) score each candidates by cosine similarity to the centroid (semantic fit) multiplied by how concentrated the term is in that cluster versus the whole corpus (penalizing generic words).

The highest-scoring term becomes the label, a label_debug DataFrame lets you audit all candidates and scores.

In [15]:
feature_to_idx = {term: i for i, term in enumerate(feature_names)}
global_dfreq = np.asarray((tfidf_matrix > 0).sum(axis=0)).ravel()  # doc freq per term

def rank_theme_labels(
    theme_to_terms,
    docs_df,
    tfidf_matrix,
    embeddings,
    model,
    feature_to_idx,
    global_dfreq,
    min_chars=3,
    pos_filter=False,  # set True to use spaCy en_core_web_sm for POS filtering
):
    """
    Description:
        For each theme, rank candidate terms using a combined score of semantic alignment
        with the cluster centroid and distinctiveness based on local vs global document frequency.
    
    Params:
        theme_to_terms: dict mapping theme_id to list of candidate terms
        docs_df: dataframe with documents and their theme_ids
        tfidf_matrix: sparse matrix of TF-IDF values for all documents
        embeddings: numpy array of shape (n_docs, embedding_dim) with document embeddings
        model: SentenceTransformer model for encoding terms
        feature_to_idx: dict mapping term to its index in the TF-IDF matrix
        global_dfreq: numpy array of shape (n_features,) with global document frequency per term
        min_chars: minimum number of characters for a term to be considered
        pos_filter: whether to filter terms by part-of-speech (requires spaCy)
        
    Returns:
        winners: dict mapping theme_id to the best label and its score
        label_debug: dataframe with all candidate terms and their scores for inspection
    
    """
    nlp = None
    if pos_filter:
        try:
            import spacy
            nlp = spacy.load("en_core_web_sm")
        except Exception as e:
            print("spaCy not available, skipping POS filter:", e)
            pos_filter = False

    def keep_term(term):
        if len(term) < min_chars:
            return False
        if not pos_filter:
            return True
        doc = nlp(term)
        return any(tok.pos_ in {"NOUN", "PROPN", "ADJ"} for tok in doc)

    rows = []
    winners = {}

    for theme_id, raw_candidates in sorted(theme_to_terms.items()):
        # clean + dedupe + ensure the term is in the vocabulary
        candidates = []
        seen = set()
        for t in raw_candidates:
            t = t.strip()
            if t in seen or t not in feature_to_idx:
                continue
            if keep_term(t):
                candidates.append(t)
                seen.add(t)

        mask = (docs_df["theme_id"] == theme_id).values
        if not mask.any() or not candidates:
            continue

        # cluster centroid in embedding space
        centroid = embeddings[mask].mean(axis=0)
        centroid /= np.linalg.norm(centroid) + 1e-12

        # candidate embeddings
        cand_embs = model.encode(candidates, convert_to_numpy=True, normalize_embeddings=True)

        # cluster-specific doc freq
        cluster_dfreq = np.asarray((tfidf_matrix[mask] > 0).sum(axis=0)).ravel()

        best_term, best_score = None, -np.inf
        for term, term_emb in zip(candidates, cand_embs):
            idx = feature_to_idx[term]
            local_freq = cluster_dfreq[idx]
            global_freq = global_dfreq[idx]
            distinctiveness = local_freq / (global_freq + 1e-6)  # step 4: penalize generic words
            sim = float(np.dot(term_emb, centroid))             # step 3: semantic alignment
            score = sim * distinctiveness                       # combined score

            rows.append(
                {
                    "theme_id": theme_id,
                    "term": term,
                    "sim": sim,
                    "local_freq": local_freq,
                    "global_freq": global_freq,
                    "distinctiveness": distinctiveness,
                    "score": score,
                }
            )
            if score > best_score:
                best_term, best_score = term, score

        winners[theme_id] = {"label": best_term, "score": best_score}

    label_debug = pd.DataFrame(rows).sort_values(["theme_id", "score"], ascending=[True, False])
    return winners, label_debug

labels_best, label_debug = rank_theme_labels(
    theme_to_terms,
    docs_with_themes,
    tfidf_matrix,
    embeddings,
    model,
    feature_to_idx,
    global_dfreq,
    min_chars=3,
    pos_filter=False,  # flip to True if spaCy is installed
)

print("Chosen labels:")
for tid, info in sorted(labels_best.items()):
    print(f"Theme {tid:02d}: {info['label']} (score={info['score']:.3f})")

# Inspect candidates and their scores if you want to review:
label_debug.head()


Chosen labels:
Theme 00: gluten free (score=0.759)
Theme 01: tell time (score=0.435)
Theme 02: casual friday (score=0.460)
Theme 03: puppy (score=0.605)
Theme 04: howl (score=0.304)
Theme 05: elephant room (score=0.566)
Theme 06: swear (score=0.114)
Theme 07: insurance cover (score=0.677)
Theme 08: funny (score=0.037)
Theme 09: perspective (score=0.241)
Theme 10: thumb (score=0.446)
Theme 11: wifi (score=0.270)
Theme 12: flight (score=0.555)
Theme 13: town (score=0.265)
Theme 14: clown (score=0.629)
Theme 15: clothing (score=0.646)
Theme 16: conversation (score=0.560)
Theme 17: climate change (score=0.794)
Theme 18: noise (score=0.497)
Theme 19: mouse (score=0.665)
Theme 20: jump (score=0.195)
Theme 21: death lemon (score=0.486)
Theme 22: tell (score=0.150)
Theme 23: relationship (score=0.403)
Theme 24: election (score=0.656)
Theme 25: netflix (score=0.552)
Theme 26: dinner (score=0.594)
Theme 27: asteroid (score=0.447)
Theme 28: france (score=0.367)
Theme 29: restaurant (score=0.615)


,theme_id,term,sim,local_freq,global_freq,distinctiveness,score
2,0,gluten free,0.775074,3382,3452,0.979722,0.759357
0,0,gluten,0.726666,4808,4916,0.978031,0.710701
1,0,vegan,0.775480,3517,4270,0.823653,0.638727
4,0,vegetarian,0.730684,1116,1568,0.711735,0.520053
7,0,lactose,0.566332,546,648,0.842593,0.477187


## 5. Saving results to csv

Export the new dataframe with themes and also a "themes.csv" files containing all themes with mean stats from the dataframe.

In [ ]:
####################################
# Map chosen labels to docs (fall back to "miscellaneous" if any theme is missing)
id_to_label = {tid: info["label"] for tid, info in labels_best.items()}
docs_with_themes["theme_label"] = (
    docs_with_themes["theme_id"].map(id_to_label).fillna("miscellaneous").str.title()
)
# Export docs_with_themes with theme_label
cols_export = [
    "doc_id", "contest_id", "kind", "text", "clean_text", "rank", "mean", "precision",
    "votes", "not_funny", "somewhat_funny", "funny", "theme_id", "theme_label",
]
# restore original text and clean_text columns
docs_with_themes["text"] = original_docs_df["text"]
docs_with_themes["clean_text"] = original_docs_df["clean_text"]

docs_with_themes.to_csv(OUTPUT_DIR / "docs_with_themes.csv", columns=cols_export, index=False)


####################################
# Build themes.csv file
themes_summary = (
    docs_with_themes
    .groupby("theme_id")
    .agg(
        nb_instances=("doc_id", "count"),
        mu_rank=("rank", "mean"),
        mu_mean=("mean", "mean"),
        mu_precision=("precision", "mean"),
        mu_not_funny=("not_funny", "mean"),
        mu_somewhat_funny=("somewhat_funny", "mean"),
        mu_funny=("funny", "mean"),
    )
    .reset_index()
)
themes_summary["theme_label"] = themes_summary["theme_id"].map(id_to_label).fillna("miscellaneous").str.title()

# add TF-IDF terms column from 'theme_to_terms' for each themes
themes_summary["TF_IDF_terms"] = themes_summary["theme_id"].map(
    lambda tid: ", ".join(theme_to_terms.get(tid, []))
)
themes_summary = themes_summary[
    ["theme_id", "theme_label", "nb_instances", "mu_rank", "mu_mean", "mu_precision",
     "mu_not_funny", "mu_somewhat_funny", "mu_funny", "TF_IDF_terms"]
]
themes_summary.to_csv(OUTPUT_DIR / "themes.csv", index=False)


####################################
# export label_debug for inspection
label_debug.to_csv(OUTPUT_DIR / "TF_IDF_themes_scores.csv", index=False)


In [ ]:
# # count oh many each themes are assigned to documents
# theme_counts = defaultdict(int)
# for theme_id in docs_with_themes["theme_id"]:
#     theme_counts[theme_id] += 1
    
# # print 20 most frequent themes
# most_frequent = sorted(theme_counts.items(), key=lambda x: x[1], reverse=True)[:20]

# print("20 most frequent themes:")
# for theme_id, count in most_frequent:
#     print(f"Theme ID {theme_id}: count={count}")    

# # print the most 20 most FUNNY themes (with funniness score)
# print("20 most funny themes:")
# funniness_scores = {}
# for theme_id, count in most_frequent:
#     theme_docs = docs_with_themes[docs_with_themes["theme_id"] == theme_id]
#     funny_mean = theme_docs["funny"].mean()
#     funniness_scores[theme_id] = funny_mean
#     print(f"Theme ID {theme_id}: count={count}, funny_mean={funny_mean:.3f}")

In [ ]:
# def auto_label_from_terms(terms, max_words=3):
#     """
#     Given a list of terms, create a simple label by selecting up to max_words
#     """
#     if not terms:
#         return "miscellaneous"
#     # prefer bigrams if possible, then unigrams
#     bigrams = [t for t in terms if " " in t]
#     unigrams = [t for t in terms if " " not in t]

#     label_terms = bigrams[:max_words]
#     if len(label_terms) < max_words:
#         label_terms += unigrams[: max_words - len(label_terms)]

#     # simple title-case label
#     label = ", ".join(label_terms)
#     return label.title()

In [ ]:
# theme_to_label = {
#     tid: auto_label_from_terms(terms)
#     for tid, terms in theme_to_terms.items()
# }

# # display the 10 first themes
# for tid in list(theme_to_label.keys())[:10]:
#     print(tid, "=>", theme_to_label[tid])

# docs_with_themes["theme_label"] = docs_with_themes["theme_id"].map(theme_to_label)
# docs_with_themes.head()


In [ ]:
# # display all unique themes_label
# docs_with_themes["theme_label"].unique()

In [ ]:
# # save results to CSV
# docs_with_themes.to_csv(OUTPUT_DIR / "docs_with_themes.csv", index=False)

In [18]:
# Correct test column with original description and add clean_text
original_csv = pd.read_csv("/Users/jeremyserillon/Desktop/ada-2025-project-json-derulo/data/newyorker_caption_contest/themes_extraction_csv_1/docs_with_clean_text.csv")
result_csv = pd.read_csv("/Users/jeremyserillon/Desktop/ada-2025-project-json-derulo/data/newyorker_caption_contest/THEMES_RESULTS_V1/docs_with_themes.csv")

In [19]:
original_csv.head()

,doc_id,contest_id,kind,text,rank,mean,precision,votes,not_funny,somewhat_funny,funny,clean_text
0,image_510,510,image,A man is relaxing on a city street. Others are...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,man relax city street go business step pass ma...
1,caption_510_0,510,caption,I'm a congressman--obstruction is my job.,0.0,1.913043,0.094022,69.0,24.0,27.0,18.0,congressman obstruction job
2,caption_510_1,510,caption,"I'm what they mean when they say, 'The middle ...",1.0,1.842105,0.191381,19.0,8.0,6.0,5.0,mean middle class step
3,caption_510_2,510,caption,Does this suit make me look flat?,2.0,1.711111,0.112915,45.0,21.0,16.0,8.0,suit look flat
4,caption_510_3,510,caption,"When the right woman comes along, I'll know it.",3.0,1.625000,0.116657,32.0,15.0,14.0,3.0,right woman come know


In [20]:
result_csv.head()

,doc_id,contest_id,kind,text,rank,mean,precision,votes,not_funny,somewhat_funny,funny,theme_id,theme_label
0,image_510,510,image,man relax city street go business step pass ma...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13,Town
1,caption_510_0,510,caption,congressman obstruction job,0.0,1.913043,0.094022,69.0,24.0,27.0,18.0,24,Election
2,caption_510_1,510,caption,mean middle class step,1.0,1.842105,0.191381,19.0,8.0,6.0,5.0,77,Stock
3,caption_510_2,510,caption,suit look flat,2.0,1.711111,0.112915,45.0,21.0,16.0,8.0,15,Clothing
4,caption_510_3,510,caption,right woman come know,3.0,1.625000,0.116657,32.0,15.0,14.0,3.0,23,Relationship


In [ ]:
# cols_export = [
#     "doc_id", "contest_id", "kind", "text", "clean_text", "rank", "mean", "precision",
#     "votes", "not_funny", "somewhat_funny", "funny", "theme_id", "theme_label",
# ]

# result_csv["text"] = original_csv["text"]
# result_csv["clean_text"] = original_csv["clean_text"]

# # change columns order
# result_csv = result_csv[cols_export]

# # save with correct columns order (same as in cols_export)
# result_csv.to_csv("/Users/jeremyserillon/Downloads/docs_with_themes.csv", columns=cols_export, index=False)

# # display modified result
# result_csv.head()


# # check if export is correct
# exported_csv = pd.read_csv("/Users/jeremyserillon/Downloads/docs_with_themes.csv")
# result_csv.equals(exported_csv)